# 实验 3：家族树关系补全

## 任务理解

输入由“人物”和“亲属关系”组成，模型输出满足该关系的人物。例如：

```text
(Colin, mother) -> Victoria
(Colin, aunt)   -> Jennifer, Margaret
```

共有 24 个人物、12 种关系和 104 个查询。一个查询可能有多个答案，因此输出使用 24 维 multi-hot 表示。

## 理论实现

人物与关系分别经过 6 维编码，再合并完成关系推理：

```text
person:   24 -> 6 ┐
                  ├-> 12 -> 6 -> 24
relation: 12 -> 6 ┘
```

每层使用 Sigmoid，24 个输出分别表示各人物是否为答案。训练采用加权二元交叉熵和 Adam；这保留论文的数据表示与网络结构，同时避免稀疏多标签目标使输出全部趋近于 0。

## 模型代码

In [1]:
from collections import defaultdict

import numpy as np

PEOPLE = [
    "Christopher", "Penelope", "Andrew", "Christine",
    "Margaret", "Arthur", "Victoria", "James",
    "Jennifer", "Charles", "Colin", "Charlotte",
    "Roberto", "Maria", "Piero", "Francesca",
    "Gina", "Emilio", "Lucia", "Marco",
    "Angela", "Tomaso", "Alfonso", "Sophia",
]
RELATIONS = [
    "father", "mother", "husband", "wife",
    "son", "daughter", "brother", "sister",
    "uncle", "aunt", "nephew", "niece",
]
MALE = {
    "Christopher", "Andrew", "Arthur", "James", "Charles", "Colin",
    "Roberto", "Piero", "Emilio", "Marco", "Tomaso", "Alfonso",
}
GENDER = {person: ("M" if person in MALE else "F") for person in PEOPLE}
COUPLES = [
    ("Christopher", "Penelope"), ("Andrew", "Christine"),
    ("Arthur", "Margaret"), ("James", "Victoria"),
    ("Charles", "Jennifer"), ("Roberto", "Maria"),
    ("Piero", "Francesca"), ("Emilio", "Gina"),
    ("Marco", "Lucia"), ("Tomaso", "Angela"),
]
PARENT_GROUPS = [
    ("Christopher", "Penelope", ["Arthur", "Victoria"]),
    ("Andrew", "Christine", ["James", "Jennifer"]),
    ("James", "Victoria", ["Colin", "Charlotte"]),
    ("Roberto", "Maria", ["Emilio", "Lucia"]),
    ("Piero", "Francesca", ["Marco", "Angela"]),
    ("Marco", "Lucia", ["Alfonso", "Sophia"]),
]

spouse_of = {}
for husband, wife in COUPLES:
    spouse_of[husband] = wife
    spouse_of[wife] = husband

parents_of = {}
siblings_of = defaultdict(set)
for father, mother, children in PARENT_GROUPS:
    for child in children:
        parents_of[child] = (father, mother)
        siblings_of[child].update(other for other in children if other != child)

facts: set[tuple[str, str, str]] = set()
for husband, wife in COUPLES:
    facts.update({(wife, "husband", husband), (husband, "wife", wife)})

for father, mother, children in PARENT_GROUPS:
    for child in children:
        facts.update({(child, "father", father), (child, "mother", mother)})
        relation = "son" if GENDER[child] == "M" else "daughter"
        facts.update({(father, relation, child), (mother, relation, child)})

for person, siblings in siblings_of.items():
    for sibling in siblings:
        relation = "brother" if GENDER[sibling] == "M" else "sister"
        facts.add((person, relation, sibling))

for child, parents in parents_of.items():
    for parent in parents:
        for blood_relative in siblings_of[parent]:
            for relative in (blood_relative, spouse_of[blood_relative]):
                relation = "uncle" if GENDER[relative] == "M" else "aunt"
                inverse = "nephew" if GENDER[child] == "M" else "niece"
                facts.update({(child, relation, relative), (relative, inverse, child)})

answers_by_query = defaultdict(set)
for person, relation, answer in facts:
    answers_by_query[(person, relation)].add(answer)

person_index = {person: index for index, person in enumerate(PEOPLE)}
relation_index = {relation: index for index, relation in enumerate(RELATIONS)}
CASES = sorted(
    answers_by_query,
    key=lambda query: (person_index[query[0]], relation_index[query[1]]),
)

person_inputs = np.zeros((len(CASES), len(PEOPLE)), dtype=np.float64)
relation_inputs = np.zeros((len(CASES), len(RELATIONS)), dtype=np.float64)
targets = np.zeros((len(CASES), len(PEOPLE)), dtype=np.float64)
for row, (person, relation) in enumerate(CASES):
    person_inputs[row, person_index[person]] = 1.0
    relation_inputs[row, relation_index[relation]] = 1.0
    for answer in answers_by_query[(person, relation)]:
        targets[row, person_index[answer]] = 1.0

HOLDOUT_QUERIES = [
    ("Colin", "mother"),
    ("Charlotte", "father"),
    ("Alfonso", "mother"),
    ("Sophia", "father"),
]
holdout_indices = np.array([CASES.index(query) for query in HOLDOUT_QUERIES])
holdout_set = set(holdout_indices.tolist())
train_indices = np.array([
    index for index in range(len(CASES)) if index not in holdout_set
])

assert len(facts) == 112
assert len(CASES) == 104
assert len(train_indices) == 100 and len(holdout_indices) == 4

In [2]:
def sigmoid(x: np.ndarray) -> np.ndarray:
    x = np.asarray(x, dtype=np.float64)
    output = np.empty_like(x)
    positive = x >= 0
    output[positive] = 1.0 / (1.0 + np.exp(-x[positive]))
    exp_x = np.exp(x[~positive])
    output[~positive] = exp_x / (1.0 + exp_x)
    return output


class FamilyTreeNetwork:
    SHAPES = {
        "W_person": (24, 6), "b_person": (6,),
        "W_relation": (12, 6), "b_relation": (6,),
        "W_central": (12, 12), "b_central": (12,),
        "W_hidden": (12, 6), "b_hidden": (6,),
        "W_output": (6, 24), "b_output": (24,),
    }

    def __init__(self, seed: int, init_scale: float = 0.3) -> None:
        rng = np.random.default_rng(seed)
        self.params = {
            name: rng.uniform(-init_scale, init_scale, size=shape)
            for name, shape in self.SHAPES.items()
        }
        self.grads = {
            name: np.zeros_like(value) for name, value in self.params.items()
        }
        self.m = {
            name: np.zeros_like(value) for name, value in self.params.items()
        }
        self.v = {
            name: np.zeros_like(value) for name, value in self.params.items()
        }
        self.t = 0
        self.cache: dict[str, np.ndarray] = {}

    def forward(
        self,
        person_batch: np.ndarray,
        relation_batch: np.ndarray,
        *,
        store_cache: bool = False,
    ) -> np.ndarray:
        person_batch = np.asarray(person_batch, dtype=np.float64)
        relation_batch = np.asarray(relation_batch, dtype=np.float64)
        if person_batch.ndim != 2 or person_batch.shape[1] != 24:
            raise ValueError("人物输入形状应为 (样本数, 24)")
        if relation_batch.ndim != 2 or relation_batch.shape[1] != 12:
            raise ValueError("关系输入形状应为 (样本数, 12)")

        person_code = sigmoid(
            person_batch @ self.params["W_person"] + self.params["b_person"]
        )
        relation_code = sigmoid(
            relation_batch @ self.params["W_relation"] + self.params["b_relation"]
        )
        combined = np.concatenate([person_code, relation_code], axis=1)
        central = sigmoid(
            combined @ self.params["W_central"] + self.params["b_central"]
        )
        hidden = sigmoid(
            central @ self.params["W_hidden"] + self.params["b_hidden"]
        )
        output = sigmoid(
            hidden @ self.params["W_output"] + self.params["b_output"]
        )

        if store_cache:
            self.cache = {
                "person_input": person_batch,
                "relation_input": relation_batch,
                "person_code": person_code,
                "relation_code": relation_code,
                "combined": combined,
                "central": central,
                "hidden": hidden,
                "output": output,
            }
        return output

    def loss_and_backward(
        self,
        person_batch: np.ndarray,
        relation_batch: np.ndarray,
        target: np.ndarray,
        *,
        positive_weight: float,
        l2: float,
    ) -> float:
        output = self.forward(person_batch, relation_batch, store_cache=True)
        cache = self.cache
        weights = np.where(target == 1.0, positive_weight, 1.0)
        delta_output = (output - target) * weights / len(target)

        self.grads["W_output"] = cache["hidden"].T @ delta_output
        self.grads["b_output"] = delta_output.sum(axis=0)

        delta_hidden = delta_output @ self.params["W_output"].T
        delta_hidden *= cache["hidden"] * (1.0 - cache["hidden"])
        self.grads["W_hidden"] = cache["central"].T @ delta_hidden
        self.grads["b_hidden"] = delta_hidden.sum(axis=0)

        delta_central = delta_hidden @ self.params["W_hidden"].T
        delta_central *= cache["central"] * (1.0 - cache["central"])
        self.grads["W_central"] = cache["combined"].T @ delta_central
        self.grads["b_central"] = delta_central.sum(axis=0)

        delta_combined = delta_central @ self.params["W_central"].T
        delta_person = delta_combined[:, :6]
        delta_person *= cache["person_code"] * (1.0 - cache["person_code"])
        delta_relation = delta_combined[:, 6:]
        delta_relation *= cache["relation_code"] * (1.0 - cache["relation_code"])

        self.grads["W_person"] = cache["person_input"].T @ delta_person
        self.grads["b_person"] = delta_person.sum(axis=0)
        self.grads["W_relation"] = cache["relation_input"].T @ delta_relation
        self.grads["b_relation"] = delta_relation.sum(axis=0)

        for name in self.params:
            if name.startswith("W_"):
                self.grads[name] += l2 * self.params[name]

        clipped = np.clip(output, 1e-12, 1.0 - 1e-12)
        data_loss = -np.mean(np.sum(
            positive_weight * target * np.log(clipped)
            + (1.0 - target) * np.log(1.0 - clipped),
            axis=1,
        ))
        penalty = 0.5 * l2 * sum(
            np.sum(value ** 2)
            for name, value in self.params.items()
            if name.startswith("W_")
        )
        return float(data_loss + penalty)

    def adam_step(
        self,
        learning_rate: float,
        beta1: float = 0.9,
        beta2: float = 0.999,
        epsilon: float = 1e-8,
    ) -> None:
        self.t += 1
        for name in self.params:
            gradient = self.grads[name]
            self.m[name] = beta1 * self.m[name] + (1.0 - beta1) * gradient
            self.v[name] = beta2 * self.v[name] + (1.0 - beta2) * gradient ** 2
            m_hat = self.m[name] / (1.0 - beta1 ** self.t)
            v_hat = self.v[name] / (1.0 - beta2 ** self.t)
            self.params[name] -= learning_rate * m_hat / (
                np.sqrt(v_hat) + epsilon
            )

In [3]:
TRAIN_CONFIG = {
    "seed": 8,
    "epochs": 5_000,
    "learning_rate": 0.01,
    "positive_weight": 3.0,
    "l2": 1e-5,
}

family_model = FamilyTreeNetwork(TRAIN_CONFIG["seed"])
for _ in range(TRAIN_CONFIG["epochs"]):
    family_model.loss_and_backward(
        person_inputs[train_indices],
        relation_inputs[train_indices],
        targets[train_indices],
        positive_weight=TRAIN_CONFIG["positive_weight"],
        l2=TRAIN_CONFIG["l2"],
    )
    family_model.adam_step(TRAIN_CONFIG["learning_rate"])

train_output = family_model.forward(
    person_inputs[train_indices], relation_inputs[train_indices]
)
holdout_output = family_model.forward(
    person_inputs[holdout_indices], relation_inputs[holdout_indices]
)
train_exact = np.all((train_output >= 0.5) == targets[train_indices], axis=1)
holdout_exact = np.all(
    (holdout_output >= 0.5) == targets[holdout_indices], axis=1
)
assert int(train_exact.sum()) == 100
assert int(holdout_exact.sum()) == 4

## 测试结果

只需修改下面的 `test_queries`。每一项写成 `(人物, 关系)`；可用名称见模型代码中的 `PEOPLE` 和 `RELATIONS`。

In [4]:
# 修改这里的输入数据
test_queries = [
    ("Colin", "mother"),
    ("Colin", "aunt"),
    ("Sophia", "father"),
]

for person, relation in test_queries:
    if person not in person_index:
        raise ValueError(f"未知人物: {person}")
    if relation not in relation_index:
        raise ValueError(f"未知关系: {relation}")

person_test = np.zeros((len(test_queries), len(PEOPLE)), dtype=np.float64)
relation_test = np.zeros((len(test_queries), len(RELATIONS)), dtype=np.float64)
for row, (person, relation) in enumerate(test_queries):
    person_test[row, person_index[person]] = 1.0
    relation_test[row, relation_index[relation]] = 1.0

scores = family_model.forward(person_test, relation_test)
print(f"训练集 exact match: {int(train_exact.sum())}/100")
print(f"留出集 exact match: {int(holdout_exact.sum())}/4")
for query, row_scores in zip(test_queries, scores):
    predicted_indices = np.flatnonzero(row_scores >= 0.5)
    predicted_indices = predicted_indices[
        np.argsort(row_scores[predicted_indices])[::-1]
    ]
    predicted = [
        f"{PEOPLE[index]} ({row_scores[index]:.4f})"
        for index in predicted_indices
    ]
    expected = sorted(answers_by_query.get(query, set()))
    print(f"{query} -> 预测: {predicted or ['无']}；已知答案: {expected or ['无']}")

训练集 exact match: 100/100
留出集 exact match: 4/4
('Colin', 'mother') -> 预测: ['Victoria (0.9770)']；已知答案: ['Victoria']
('Colin', 'aunt') -> 预测: ['Margaret (0.9958)', 'Jennifer (0.9855)']；已知答案: ['Jennifer', 'Margaret']
('Sophia', 'father') -> 预测: ['Marco (0.9911)']；已知答案: ['Marco']
